In [16]:
from dotenv import load_dotenv
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool


In [17]:
from langchain.chat_models import init_chat_model
import os



In [22]:
model = init_chat_model(
    model="qwen3.5-plus",  # 这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [23]:
from langchain_tavily import TavilySearch
#web搜索
web_search = TavilySearch(
    max_results=5,
    topic="general",
)

In [24]:
from langgraph.checkpoint.sqlite import SqliteSaver
import _sqlite3
import os

# 关键：如果resources文件夹不存在，就新建
os.makedirs("resources", exist_ok=True)

In [25]:
from langgraph.checkpoint.sqlite import SqliteSaver
import _sqlite3
#连接sqlite
connection=_sqlite3.connect("resources/personal_chief.db",check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

In [26]:
from langchain.agents import create_agent
system_prompt="""
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作:
.识别和评估食材:若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用
食材清单”。
.智能食谱检索:优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
.多维度评估与排序:从营养价值和制作难度两个维度对检索到的候选食请进行量化打分，并根据得分排序，制作简单且营养丰富的排
名靠前。
.结构化方案输出:把排序后的食请整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食请的参考图片，帮助用户快速做出决策。
请严格按照流程，优先调用 web_search工具搜索食谱，搜索不到的情况下才能自己发挥。

"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [27]:
from langchain.messages import HumanMessage
multimodal_message = HumanMessage([
    {"type":"text","text":"你看看能帮我做什么."},
    {"type":"image","url":"https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"}
])
config = {"configurable":{"thread_id":"4"}}



In [28]:
response = agent.invoke({"messages": [multimodal_message]},config)

In [29]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image_url', 'image_url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ========================

In [30]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="我喜欢第三道菜，可以详细点吗")]
    },
    config
)

In [31]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image_url', 'image_url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '你看看能帮我做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ========================